In [4]:
from langserve import add_routes


In [5]:
from google.cloud import aiplatform

In [6]:
project_id = "function-health-dev-env"  # @param {type:"string"}
database_password = "FunctionHealth"  # @param {type:"string"}
region = "us-central1"  # @param {type:"string"}
instance_name = "development-ac-poc"  # @param {type:"string"}
database_name = "poc"  # @param {type:"string"}
database_user = "ac-dev"  # @param {type:"string"}

In [7]:
# enable Vertex AI API
!gcloud services enable aiplatform.googleapis.com

In [8]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=hhzTEXEWWGBBl86Vb3qtjZobFvZTRf&access_type=offline&code_challenge=gTEo7YvS1Att89mCv0gRaQwEQTpNpthyh6HL_qm5zMM&code_challenge_method=S256


Credentials saved to file: [/Users/andrea/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "function-health-dev-env" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [9]:
project_id = "function-health-dev-env"  # @param {type:"string"}
database_password = "FunctionHealth"  # @param {type:"string"}
region = "us-central1"  # @param {type:"string"}
instance_name = "development-ac-poc"  # @param {type:"string"}
database_name = "poc"  # @param {type:"string"}
database_user = "ac-dev"  # @param {type:"string"}


In [10]:
!gcloud services enable \
  bigquery.googleapis.com \
  sqladmin.googleapis.com \
  aiplatform.googleapis.com \
  cloudresourcemanager.googleapis.com \
  artifactregistry.googleapis.com \
  cloudbuild.googleapis.com \
  run.googleapis.com \
  secretmanager.googleapis.com

Operation "operations/acat.p2-1014551664922-0c493c31-09e9-4926-b49c-823e433c29d0" finished successfully.


In [42]:
from typing import List

from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
import asyncio
import asyncpg
from google.cloud.sql.connector import Connector
import numpy as np
from pgvector.asyncpg import register_vector
import ollama

In [40]:
import ollama

In [67]:
query = "what's good for high cholesterol"

In [41]:
def embed_query(query:str):
    qe = ollama.embeddings(model="mxbai-embed-large", prompt=query)
    qe1 = qe['embedding']
    return(qe1)

In [68]:
qe1 = embed_query(query)

In [83]:

async def main(qe1):
    matches = []
    loop = asyncio.get_running_loop()
    async with Connector(loop=loop) as connector:
        # Create connection to Cloud SQL database.
        conn: asyncpg.Connection = await connector.connect_async(
            f"{project_id}:{region}:{instance_name}",  # Cloud SQL instance connection name
            "asyncpg",
            user=f"{database_user}",
            password=f"{database_password}",
            db=f"{database_name}",
        )

        await register_vector(conn)
        similarity_threshold = 0.3
        num_matches = 10

        # Find similar products to the query using cosine similarity search
        # over all vector embeddings. This new feature is provided by `pgvector`.
        results = await conn.fetch(
            """
                            WITH vector_matches AS (
                              SELECT content, 1 - (embedding <=> $1) AS similarity
                              FROM data_set1
                              WHERE 1 - (embedding <=> $1) > $2
                              ORDER BY similarity DESC
                              LIMIT $3
                            )
                            SELECT * from vector_matches
                            """,
            qe1,
            similarity_threshold,
            num_matches,
    
        )

        if len(results) == 0:
            raise Exception("Did not find any results. Adjust the query parameters.")

        for r in results:
            # Collect the description for all the matched similar toy products.
            matches.append(
                f"""{r["content"]}.
                         ."""
            )
        await conn.close()
        return(matches)



In [84]:
await main(qe1) 

['if\nLDL\ncholesterol\nis\nhigh,\nif.\n                         .',
 'on\na\nfull\nrisk\nassessment,\nas\nnoted\nabove,\nwhich\nshould\ninclude\nthe \nquality\nand\nnumber\nof\ncholesterol\nparticles\nand\nother\nrisk\nfactors.\nSince\ncholesterol\nis\nan\nessential\nbuilding\nblock\nof\ncells,\nthe\nbrain,\nnerves,\nand\nhormones, \nlowering\ncholesterol\ntoo\nmuch\nmay\nhave\nnegative\nconsequences.\nUntil\nrecently,\nwe\nhave\nnot.\n                         .',
 'supplements\nmay\nhelp\nimprove\ncardiovascular\nhealth\nand\nlipid\nprofiles.\nHowever,\nit \nis\nimportant\nto\naddress\nthe\nroot\ncause\nor\ncauses\nof\nabnormal\nlipids.\n●\nOmega-3\nRejuvenate\nby\nBig\nBold\nHealth,\ntwice\ndaily \n●\nPGX\nby\nNatural\nFactors\nPowder\n5\ngrams\nin\nwater\nbefore\nmeals\n(fiber) \n●\nRed\nYeast\nRice\n+\nCoQ10\n(natural\nstatin-like\nmolecule)\nby\nThorne\n600\nmg,\ntwice\ndaily \n●\nMeta-Sitosterol\n(plant\nsterols)\nby\nMetagenics,\none\nwith\neach\nmeal \n●\nArterosil\nHP\nby\nDe

In [87]:
from langchain.chains.summarize import load_summarize_chain
from langchain.docstore.document import Document
from langchain import PromptTemplate, LLMChain
from IPython.display import display, Markdown
from langchain_google_vertexai import ChatVertexAI

llm = ChatVertexAI(model_name="gemini-pro")

map_prompt_template = """
              You will be given a detailed description of health realted concepts
              This description is enclosed in triple backticks (```).
            

              ```{text}```
              SUMMARY:
              """
map_prompt = PromptTemplate(template=map_prompt_template, input_variables=["text"])

In [88]:
user_query = "what does hdl cholesterol do?"

In [89]:
qe1 = embed_query(query)
matches = await main(qe1)

In [90]:
from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

In [101]:
user_query = "what does hdl cholesterol do?"

In [105]:
combine_prompt_template = """
                You will be given health information
                enclosed in triple backticks (```) and a question enclosed in
                double backticks(``).
                You are a medical professional. Please answer the question in 200 words in an empathetic manner. Use only the given health information to answer


                Description:
                ```{text}```


                Question:
                ``{user_query}``


                Answer:
                """
combine_prompt = PromptTemplate(
    template=combine_prompt_template, input_variables=["text", "user_query"]
)
def embed_query(user_query:str):
    qe = ollama.embeddings(model="mxbai-embed-large", prompt=user_query)
    qe1 = qe['embedding']
    return(qe1)

async def get_docs(user_query:str):
    qe1 = embed_query(user_query)
    matches = await main(qe1)
    return [Document(page_content=t) for t in matches]
    
user_query = "what does hdl cholesterol do?"    
docs = await get_docs(user_query)

chain = load_summarize_chain(
    llm, chain_type="map_reduce", map_prompt=map_prompt, combine_prompt=combine_prompt
)
#chain1 =  ({
 #       RunnableLambda(embed_query) | RunnableLambda(main) |
  #      chain
   # })
answer = chain.run(
    {
        "input_documents": docs,
        "user_query": user_query,
    }
)


display(Markdown(answer))

## Understanding HDL Cholesterol: Your Body's Clean-up Crew

HDL, or High-Density Lipoprotein cholesterol, plays a crucial role in keeping your heart healthy. Imagine it like a tiny garbage truck that travels through your bloodstream, collecting excess cholesterol from your arteries and transporting it back to your liver for processing and elimination. 

Think of LDL cholesterol as the "bad cholesterol" because it contributes to plaque buildup in your arteries, increasing your risk of heart attack or stroke. HDL, on the other hand, acts as the "good cholesterol" by:

**1. Removing Excess Cholesterol:**  HDL acts like a vacuum cleaner for your arteries, removing the excess LDL cholesterol that can cause blockages and lead to cardiovascular problems. 
**2. Preventing Plaque Formation:** By keeping your arteries clean and free of cholesterol buildup, HDL reduces the chances of developing plaque, which is a major risk factor for heart attacks and strokes.
**3. Reducing Inflammation:**  HDL also has anti-inflammatory properties, which helps protect your arteries and further reduces your risk of cardiovascular diseases. 

**The higher your HDL cholesterol level, the lower your risk of heart disease.** This means that maintaining healthy HDL levels is essential for protecting your heart and overall well-being.

If you have any concerns about your HDL levels or heart health, don't hesitate to talk to your doctor. They can provide personalized advice on managing your cholesterol and keeping your heart healthy for years to come. 


In [104]:
await get_docs(user_query)